# Weighted Cosine Similarity

**Easy** &nbsp;·&nbsp; TensorTonic &nbsp;·&nbsp; `Linear Algebra`

Plain cosine similarity treats every feature as equally important. In a real
recommender that is rarely true: for "how similar are these two films" the
*genre* column should count for more than the *runtime in minutes* column, and
you know that in advance.

Weighted cosine similarity attaches a weight $w_i \ge 0$ to each feature:

$$\text{weighted\_cosine}(a, b, w) =
\frac{\sum_i w_i\, a_i b_i}
{\sqrt{\sum_i w_i\, a_i^2} \; \sqrt{\sum_i w_i\, b_i^2}}$$

Set $w_i = 1$ everywhere and every term collapses back into ordinary cosine
similarity.

---

**Example 1:**

```
Input:  a = [1, 1], b = [1, 0], w = [1, 1]
Output: 0.7071067811865475
Both features count the same -> this is plain cosine, 45 degrees.
```

**Example 2:**

```
Input:  a = [1, 1], b = [1, 0], w = [3, 1]
Output: 0.8660254037844387
Feature 0 - the one they agree on - now counts triple, so they look
more alike. Same vectors, different verdict.
```

**Example 3:**

```
Input:  a = [1, 1], b = [1, 0], w = [1, 3]
Output: 0.5
Weight the feature they DISAGREE on instead, and they look less alike.
```

---

**Hint 1:** every sum has the shape $\sum_i w_i x_i y_i$. Write that once as a
helper, `weighted_dot(x, w, y)`, and the function is three calls and a division.

**Hint 2:** you have seen this shape before. This is
[Soft Cosine Similarity](SoftCosineSimilarity.ipynb) with `S = np.diag(w)` --
all the off-diagonal terms are zero, so the double sum becomes a single one.

**Requirements:**

- input: 1-D NumPy arrays `a`, `b`, `w`, all of length `n`
- output: scalar float
- fully vectorized (no Python loops over features)
- return `0.0` if either denominator term is zero
- with `w = np.ones(n)` the result must equal plain cosine similarity

**Constraints:**

- `len(a) == len(b) == len(w) <= 10^4`
- `w_i >= 0`
- use only NumPy

---

You already have `dot_product`, `norm` and `cosine_similarity` from the earlier
problems. This is the same three lines with `w` slipped inside every sum.

### What is actually new here

Almost nothing, and that is the point. The dot product $x \cdot y$ has become
$\sum_i w_i x_i y_i$, and that single change propagates everywhere:

- $a \cdot b \;\rightarrow\; \sum_i w_i a_i b_i$
- $\|a\| = \sqrt{a \cdot a} \;\rightarrow\; \sqrt{\sum_i w_i a_i^2}$

`np.sum(w * x * y)` is the whole helper. If you are writing `for i in range(n)`,
stop.

### Trap 1: weighting the wrong thing

There are two places you could multiply by `w`, and only one of them is this
formula. The wrong one is seductive because it reads more naturally:

```
a_hat = a / norm(a)          # normalise FIRST
b_hat = b / norm(b)
return np.sum(w * a_hat * b_hat)      # ...then weight
```

That is **not** weighted cosine. The weights have to be inside the norms too,
or you are no longer dividing by the length of the thing you took the dot
product of. Run both on Example 2 before you look: one returns
`0.8660254037844387`, the other returns something **greater than 1**, which no
cosine of an angle is ever allowed to be. A similarity that can exceed 1 is a
bug you will not notice until it silently corrupts a ranking.

### Trap 2: predict before you run

**Does scaling every weight change the answer?** Work out
$\text{wcos}(a, b, 2w)$ on paper before the test cell prints it. The `2` comes
out of the numerator once and out of *each* square root once --
$\sqrt{2}\cdot\sqrt{2} = 2$ -- so it cancels exactly.

Consequence: `w = [1, 1]` and `w = [5, 5]` are the same weighting, and
normalising `w` to sum to 1 is cosmetic. Only the **ratios** between weights
carry information. Weights are relative, always.

### Trap 3: a negative weight

The constraints say $w_i \ge 0$, and the test cell checks what you do when that
is violated. With a negative weight, $\sum_i w_i a_i^2$ can go negative and
`np.sqrt` hands you a `nan` plus a `RuntimeWarning` -- exactly the failure you
met in [Soft Cosine](SoftCosineSimilarity.ipynb), because `np.diag(w)` is
positive semi-definite *if and only if* every `w_i >= 0`. This is the same bug
wearing a simpler costume.

Two defensible answers -- raise, or clamp -- and one indefensible one, which is
returning `nan` silently.

### A second version worth writing

Since $w_i a_i b_i = (\sqrt{w_i}\, a_i)(\sqrt{w_i}\, b_i)$, weighted cosine is
*exactly* plain cosine on the two **rescaled** vectors $\sqrt{w} \odot a$ and
$\sqrt{w} \odot b$. Write `weighted_cosine_sqrt` and confirm it agrees.

This is the Cholesky trick from the soft cosine notebook, degenerate: for a
diagonal `S`, the factor $L$ is just $\text{diag}(\sqrt{w})$. And it pays off
the same way -- rescale your whole corpus once, then serve weighted cosine at
the speed of plain cosine forever after.

### A third, for the shape you will actually use

`weighted_cosine_matrix(a, matrix, w)` -- one query against every **row** of a
2-D array, no Python loop over the rows. Think about `axis=` and the `where=`
guard for zero rows, same as before.

In [ ]:
import numpy as np


class Solution:
    def dot_product(self, x, y) -> float:
        if len(x) != len(y):
            raise ValueError("Vectors must have the same length")
        return float(np.dot(x, y))

    def norm(self, a):
        return np.sqrt(np.dot(a, a))

    def cosine_similarity(self, a, b) -> float:
        a_norm = self.norm(a)
        b_norm = self.norm(b)
        if a_norm == 0 or b_norm == 0:
            return 0.0
        return self.dot_product(a, b) / (a_norm * b_norm)

    # sum_i w_i x_i y_i - write this one first, everything else is three calls to it
    def weighted_dot(self, x, w, y) -> float:
        pass

    def weighted_cosine_similarity(self, a, b, w) -> float:
        pass

    # optional: rescale by sqrt(w) once, then it is plain cosine
    def weighted_cosine_sqrt(self, a, b, w) -> float:
        pass

    # optional: one vector vs every row of a matrix -> array of similarities
    def weighted_cosine_matrix(self, a, matrix, w):
        pass

In [ ]:
def check(got, want, tol=1e-9):
    """Compare one result against its expected value."""
    if got is None:
        return "not implemented"
    try:
        return "OK" if abs(got - want) < tol else f"WRONG got {got!r} want {want!r}"
    except TypeError:
        return f"WRONG got {got!r} (expected a number)"


def fmt_w(w):
    """[1. 3.] rather than a screenful of np.float64(...)."""
    return np.array2string(np.asarray(w, dtype=float), precision=2, separator=", ")


sol = Solution()

ONES2, ONES3 = np.ones(2), np.ones(3)

cases = [
    # (a, b, w, expected)
    ([1, 2, 3], [2, 4, 6], ONES3,        1.0),                 # w = 1 -> plain cosine
    ([1, 0],    [0, 1],    ONES2,        0.0),                 # w = 1, orthogonal
    ([1, 1],    [1, 0],    ONES2,        0.7071067811865475),  # example 1: 45 degrees
    ([1, 1],    [1, 0],    [3, 1],       0.8660254037844387),  # example 2: agree-feature up
    ([1, 1],    [1, 0],    [1, 3],       0.5),                 # example 3: disagree-feature up
    ([1, 0],    [-1, 0],   [2, 5],      -1.0),                 # opposite, any weights
    ([1, 2, 3], [1, 2, 3], [0.1, 7, 2],  1.0),                 # a vs itself -> 1.0
    ([1, 2, 3], [3, 2, 1], ONES3,        0.7142857142857143),  # w = 1, partial overlap
    ([1, 2, 3], [3, 2, 1], [10, 1, 1],   0.7915457172050326),  # feature 0 dominates
    ([0, 0],    [1, 2],    ONES2,        0.0),                 # zero vector -> 0.0, NOT nan
    ([0, 0],    [0, 0],    ONES2,        0.0),                 # both zero
    ([1, 2],    [1, 2],    [0, 0],       0.0),                 # all weights zero -> 0.0
]

print("weighted_cosine_similarity")
for a, b, w, want in cases:
    print(f"  {str(a):<11} {str(b):<11} w={fmt_w(w):<14}"
          f" -> {check(sol.weighted_cosine_similarity(a, b, w), want, 1e-12)}")

print("\nweighted_cosine_sqrt")
for a, b, w, want in cases:
    print(f"  {str(a):<11} {str(b):<11} w={fmt_w(w):<14}"
          f" -> {check(sol.weighted_cosine_sqrt(a, b, w), want, 1e-12)}")

rng = np.random.default_rng(11)
V = rng.normal(size=(20, 5))
W = rng.random(5) + 0.05

try:
    # w = 1 must reproduce plain cosine exactly, on random vectors not just the 12 above
    d = [abs(sol.weighted_cosine_similarity(V[i], V[j], np.ones(5)) - sol.cosine_similarity(V[i], V[j]))
         for i in range(20) for j in range(20)]
    print(f"\nw = 1 matches plain cosine on 400 pairs  : {max(d) < 1e-12}")

    # trap 2: only the RATIOS between weights matter
    scaled = [abs(sol.weighted_cosine_similarity(V[i], V[j], W)
                  - sol.weighted_cosine_similarity(V[i], V[j], W * 7.3))
              for i in range(20) for j in range(20)]
    print(f"unchanged when every weight is scaled   : {max(scaled) < 1e-12}")

    # symmetric, self-similarity exactly 1, and never outside [-1, 1]
    sym = [abs(sol.weighted_cosine_similarity(V[i], V[j], W)
               - sol.weighted_cosine_similarity(V[j], V[i], W))
           for i in range(20) for j in range(20)]
    print(f"symmetric in a and b                    : {max(sym) < 1e-12}")
    print(f"weighted_cosine(a, a) == 1              : "
          f"{max(abs(sol.weighted_cosine_similarity(V[i], V[i], W) - 1) for i in range(20)) < 1e-12}")
    in_range = all(abs(sol.weighted_cosine_similarity(V[i], V[j], W)) <= 1 + 1e-12
                   for i in range(20) for j in range(20))
    print(f"never leaves [-1, 1]                    : {in_range}")

    r = sol.weighted_cosine_similarity([1, 1], [1, 0], [3, 1])
    print(f"returns exactly float                   : {type(r) is float} (got {type(r).__name__})")
except TypeError:
    print("\nfill in weighted_cosine_similarity to run the property checks")

# trap 3: a negative weight. nan is the one answer that is not allowed.
print("\nnegative weight, w = [1, -3]:")
try:
    bad = sol.weighted_cosine_similarity([1, -1], [1, 1], [1, -3])
    if bad is None:
        print("  not implemented")
    elif bad != bad:                       # nan is the only value not equal to itself
        print("  returned nan - silently wrong. raise or clamp instead.")
    else:
        print(f"  returned {bad!r} - you clamped. defensible; say so in a comment.")
except (ValueError, AssertionError) as e:
    print(f"  raised {type(e).__name__}: {e} - defensible.")

# the matrix version: one query vs every row
print("\nweighted_cosine_matrix")
Q = np.array([1.0, 1.0])
M = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [0.0, 0.0]])
WM = np.array([3.0, 1.0])
want_rows = np.array([0.8660254037844387, 0.5, 1.0, 0.0])
got_rows = sol.weighted_cosine_matrix(Q, M, WM)
if got_rows is None:
    print("  not implemented")
else:
    got_rows = np.asarray(got_rows, dtype=float)
    ok = got_rows.shape == want_rows.shape and np.allclose(got_rows, want_rows, atol=1e-12)
    print(f"  {'OK' if ok else f'WRONG got {got_rows} want {want_rows}'}")

### After it passes: the weights are the model

Four films over the features `[genre_action, genre_drama, runtime_hours]`.
`runtime_hours` is on a completely different scale from the two genre flags,
and with equal weights it dominates both -- the metric ends up ranking films by
*length*, which nobody asked for.

Run the cell and compare the three weightings. Then notice the same thing the
soft cosine notebook ended on: the formula is four lines and it has no opinion
about anything. Every judgement in the output came out of `w`.

Which means the honest question is never "is weighted cosine better than
cosine" -- it is "where did `w` come from". Hand-set by you? Then it encodes
your assumptions, and you should be able to defend each number. Learned from
click data? Then it encodes whatever bias was in the clicks. Left at
`np.ones(n)`? That is not the absence of a choice, it is the claim that a
runtime in hours matters exactly as much as a genre flag.

In [ ]:
features = ["action", "drama", "runtime_h"]

films = {
    "Die Hard":        np.array([1.0, 0.0, 2.1]),
    "Commando":        np.array([1.0, 0.0, 1.6]),
    "Marriage Story":  np.array([0.0, 1.0, 2.2]),
}

weightings = [
    ("equal            w = [1, 1, 1]", np.array([1.0, 1.0, 1.0])),
    ("genre-first      w = [10, 10, 1]", np.array([10.0, 10.0, 1.0])),
    ("runtime-only     w = [0, 0, 1]", np.array([0.0, 0.0, 1.0])),
]

pairs = [("Die Hard", "Commando"), ("Die Hard", "Marriage Story")]

print(f"{'weighting':<34}{'action pair':>14}{'cross-genre':>14}")
for label, w in weightings:
    row = []
    for x, y in pairs:
        v = sol.weighted_cosine_similarity(films[x], films[y], w)
        row.append(f"{v:>14.4f}" if isinstance(v, float) else f"{'n/a':>14}")
    print(f"{label:<34}{''.join(row)}")

print("\ntwo action films vs an action film and a drama.")
print("equal weights barely separate them - runtime_h is the biggest number")
print("in every vector, so it drowns out both genre flags.")
print("every bit of that judgement came out of w, not out of the formula.")